In [ ]:
# CELL 1 — Install Dependencies
!pip install -q openai google-generativeai gradio requests beautifulsoup4 chromadb sentence-transformers

In [ ]:
# CELL 2 — API Keys
import os
from google.colab import userdata
from openai import OpenAI

OPENAI_API_KEY = userdata.get('OpenAPI')
os.environ["OpenAPI"] = OPENAI_API_KEY
openai_client = OpenAI(api_key=OPENAI_API_KEY)

print("OpenAI client ready.")

OpenAI client ready.


In [ ]:
# CELL 3 — RAG: Web Scraper + ChromaDB Vector Store

import requests
from bs4 import BeautifulSoup
import chromadb
from chromadb.utils import embedding_functions
import hashlib, re

# ── Embedding function (free, local sentence-transformers) ──
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# ── ChromaDB in-memory collection ──
chroma_client = chromadb.Client()
travel_collection = chroma_client.get_or_create_collection(
    name="travel_knowledge",
    embedding_function=ef,
    metadata={"hnsw:space": "cosine"},
)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; BudgetTravelScout/1.0; student-project)"
}


def scrape_wikivoyage(destination):
    """Scrape Wikivoyage for budget travel info about a destination."""
    slug = destination.strip().replace(" ", "_").title()
    url = f"https://en.wikivoyage.org/wiki/{slug}"
    chunks = []
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        if resp.status_code != 200:
            print(f"  ⚠️ Wikivoyage returned status {resp.status_code} for {destination}")
            return []
        soup = BeautifulSoup(resp.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "table"]):
            tag.decompose()
        current_heading = "Overview"
        current_text = []
        for el in soup.select("#mw-content-text .mw-parser-output > *"):
            if el.name in ("h2", "h3"):
                if current_text:
                    text = " ".join(current_text).strip()
                    if len(text) > 50:
                        chunks.append(f"[{destination} - {current_heading}] {text}")
                current_heading = el.get_text(strip=True).replace("[edit]", "")
                current_text = []
            elif el.name == "p":
                t = el.get_text(strip=True)
                if t:
                    current_text.append(t)
        if current_text:
            text = " ".join(current_text).strip()
            if len(text) > 50:
                chunks.append(f"[{destination} - {current_heading}] {text}")
    except requests.exceptions.Timeout:
        print(f"  ⚠️ Wikivoyage timed out for {destination}")
    except Exception as e:
        print(f"  ⚠️ Wikivoyage scrape failed: {e}")
    return chunks


def scrape_budget_tips(destination):
    """Scrape budget travel tips from Nomadic Matt blog."""
    slug = destination.strip().lower().replace(" ", "-")
    urls = [
        f"https://www.nomadicmatt.com/travel-guides/{slug}-travel-tips/",
        f"https://www.nomadicmatt.com/travel-guides/{slug}-on-a-budget/",
    ]
    chunks = []
    for url in urls:
        try:
            resp = requests.get(url, headers=HEADERS, timeout=10)
            if resp.status_code != 200:
                continue
            soup = BeautifulSoup(resp.text, "html.parser")
            article = soup.find("article") or soup.find("div", class_="entry-content")
            if not article:
                continue
            for tag in article(["script", "style", "aside", "figure"]):
                tag.decompose()
            for p in article.find_all("p"):
                text = p.get_text(strip=True)
                if len(text) > 60:
                    chunks.append(f"[Budget Guide - {destination}] {text}")
        except Exception as e:
            print(f"  ⚠️ Blog scrape failed for {url}: {e}")
    return chunks


def chunk_text(text, max_chars=500):
    """Split long text into overlapping chunks for better retrieval."""
    if len(text) <= max_chars:
        return [text]
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks, current = [], ""
    for sent in sentences:
        if len(current) + len(sent) > max_chars and current:
            chunks.append(current.strip())
            current = sent + " "
        else:
            current += sent + " "
    if current.strip():
        chunks.append(current.strip())
    return chunks


def build_rag_knowledge(destination):
    """Scrape multiple sources, chunk, and store in ChromaDB. Returns chunk count."""
    print(f"  🔍 Scraping Wikivoyage for {destination}...")
    wiki_chunks = scrape_wikivoyage(destination)
    print(f"    → {len(wiki_chunks)} sections from Wikivoyage")

    print(f"  🔍 Scraping budget travel blogs...")
    blog_chunks = scrape_budget_tips(destination)
    print(f"    → {len(blog_chunks)} sections from travel blogs")

    all_raw = wiki_chunks + blog_chunks
    all_chunks = []
    for raw in all_raw:
        all_chunks.extend(chunk_text(raw, max_chars=500))

    if not all_chunks:
        # ── FALLBACK: seed with basic info so ChromaDB is never empty ──
        all_chunks = [
            f"{destination} is a popular travel destination. Budget travelers can find affordable accommodation in hostels and guesthouses.",
            f"Free activities in {destination} include walking tours, visiting public parks, exploring local markets, and visiting free museums.",
            f"Budget food options in {destination} include street food, local markets, and affordable restaurants away from tourist areas.",
            f"Transportation in {destination} can be done cheaply using public transit, buses, and shared rides.",
            f"The best time to visit {destination} on a budget is during shoulder season when prices for flights and hotels are lower.",
        ]
        print(f"  ℹ️ No web data scraped — using 5 fallback seed chunks")

    # Deduplicate and add to ChromaDB
    ids, documents = [], []
    for chunk in all_chunks:
        doc_id = hashlib.md5(chunk.encode()).hexdigest()[:12]
        if doc_id not in ids:
            ids.append(doc_id)
            documents.append(chunk)

    try:
        travel_collection.upsert(ids=ids, documents=documents)
        print(f"  ✅ {len(documents)} chunks indexed in ChromaDB")
    except Exception as e:
        print(f"  ⚠️ ChromaDB upsert failed: {e}")
        return 0

    return len(documents)


def rag_query(query, n_results=5):
    """
    Retrieve top-k relevant chunks from ChromaDB for a query.
    KEY FIX: Clamps n_results to actual collection size to avoid
    ChromaDB errors when fewer docs exist than requested.
    """
    try:
        count = travel_collection.count()
        if count == 0:
            print("  ℹ️ RAG collection is empty — returning no context")
            return ""
        # ── CLAMP n_results to actual doc count ──
        safe_n = min(n_results, count)
        results = travel_collection.query(query_texts=[query], n_results=safe_n)
        docs = results["documents"][0] if results["documents"] else []
        return "\n\n".join(docs)
    except Exception as e:
        print(f"  ⚠️ RAG query failed (non-fatal): {e}")
        return ""


print("✅ RAG pipeline ready (BeautifulSoup → ChromaDB → SentenceTransformers)")

✅ RAG pipeline ready (BeautifulSoup → ChromaDB → SentenceTransformers)


In [ ]:
# CELL 4 — Agent Logic (Enhanced with RAG + Budget Constraints)

import json, re

def parse_json(raw):
    """Strip markdown fences and parse JSON safely."""
    raw = re.sub(r"```json|```", "", raw).strip()
    return json.loads(raw)


# ── Agent A — Researcher (GPT-4o + RAG) ────────────────────
def agent_a(origin, destination, max_budget=None):
    """
    GPT-4o + RAG context from scraped web data for more accurate
    flight and hotel cost estimates. Budget-aware when max_budget is set.
    Falls back gracefully if RAG returns nothing.
    """
    # Safe RAG retrieval — never crashes
    try:
        rag_context = rag_query(
            f"{destination} budget travel costs flights hotels accommodation prices",
            n_results=5,
        )
    except Exception as e:
        print(f"  ⚠️ Agent A RAG lookup failed (non-fatal): {e}")
        rag_context = ""

    rag_block = ""
    if rag_context:
        rag_block = f"\n\n=== SCRAPED WEB DATA (RAG Context) ===\n{rag_context}\n=== END ==="

    budget_instruction = ""
    if max_budget and max_budget > 0:
        budget_instruction = (
            f"\nIMPORTANT: The traveler's max daily budget is ${max_budget:.0f}. "
            "Focus on options that fit within this constraint. "
            "Flag if the destination is likely over budget."
        )

    resp = openai_client.chat.completions.create(
        model="gpt-4o",
        temperature=0.2,
        max_tokens=300,
        messages=[
            {"role": "system", "content": (
                "You are a travel cost researcher with extensive knowledge of global airfare and hotel prices. "
                + ("You have been given SCRAPED WEB DATA below to ground your estimates in real information. "
                   "Use this data to give more accurate estimates, but also use your own knowledge. "
                   if rag_context else "")
                + "Give realistic budget estimates based on typical prices. "
                f"{budget_instruction}\n"
                "Reply ONLY with a valid JSON object — no markdown, no explanation:\n"
                '{"flight_low":int,"flight_high":int,"hotel_low":int,"hotel_high":int,'
                '"booking_tip":"string","budget_warning":"string or null"}'
            )},
            {"role": "user", "content": (
                f"Estimate the typical round-trip economy flight cost from {origin} to {destination}, "
                f"and the average budget/mid-range hotel cost per night in {destination}. "
                "Give realistic USD price ranges a traveler would actually find in 2025."
                f"{rag_block}"
            )},
        ],
    )
    raw = resp.choices[0].message.content.strip()
    try:
        result = parse_json(raw)
        result.setdefault("budget_warning", None)
        return result
    except Exception:
        def extract(key):
            m = re.search(rf'"{key}"\s*:\s*(\d+)', raw)
            return int(m.group(1)) if m else 0
        tip_m = re.search(r'"booking_tip"\s*:\s*"([^"]+)"', raw)
        return {
            "flight_low":  extract("flight_low")  or 300,
            "flight_high": extract("flight_high") or 800,
            "hotel_low":   extract("hotel_low")   or 70,
            "hotel_high":  extract("hotel_high")  or 160,
            "booking_tip": tip_m.group(1) if tip_m else "Book 6-8 weeks in advance for best rates.",
            "budget_warning": None,
        }


# ── Agent B — Local Guide (GPT-4o + RAG) ────────────────────
def agent_b(destination, interest, max_budget=None):
    """
    GPT-4o + RAG context to suggest 3 free or cheap local activities.
    Budget-aware: prioritizes free activities when budget is tight.
    Falls back gracefully if RAG returns nothing.
    """
    # Safe RAG retrieval — never crashes
    try:
        rag_context = rag_query(
            f"{destination} free cheap {interest} activities things to do budget",
            n_results=5,
        )
    except Exception as e:
        print(f"  ⚠️ Agent B RAG lookup failed (non-fatal): {e}")
        rag_context = ""

    rag_block = ""
    if rag_context:
        rag_block = f"\n\n=== SCRAPED WEB DATA (RAG Context) ===\n{rag_context}\n=== END ==="

    budget_note = ""
    if max_budget and max_budget > 0:
        if max_budget < 80:
            budget_note = "The traveler is on a VERY tight budget. ALL activities should be FREE."
        elif max_budget < 150:
            budget_note = "The traveler is budget-conscious. Prioritize free options."

    resp = openai_client.chat.completions.create(
        model="gpt-4o",
        temperature=0.3,
        max_tokens=400,
        messages=[
            {"role": "system", "content": (
                "You are a savvy local travel guide. "
                + ("You have been given SCRAPED WEB DATA below with real local information. "
                   "Use it to suggest specific, real places and activities (not generic ones). "
                   if rag_context else
                   "Suggest specific, real places and activities based on your knowledge. ")
                + f"{budget_note}\n"
                "Reply ONLY with a valid JSON array — no markdown, no explanation:\n"
                '[{"name":"string","description":"one sentence","cost":"Free | Under $5 | $5-$15",'
                '"source":"where you found this info"}]'
            )},
            {"role": "user", "content": (
                f"Destination: {destination}\n"
                f"Traveler interest: {interest}\n\n"
                "List exactly 3 activities. At least 2 must be completely Free."
                f"{rag_block}"
            )},
        ],
    )
    raw = resp.choices[0].message.content.strip()
    try:
        result = parse_json(raw)
        # Guard against empty or malformed response
        if not result or not isinstance(result, list) or len(result) == 0:
            raise ValueError("Empty or invalid activity list")
        return result
    except Exception:
        return [
            {"name": "City walking tour",  "description": f"Explore {destination}'s streets and landmarks.", "cost": "Free", "source": "general"},
            {"name": "Local market visit", "description": "Browse produce, crafts, and street food.",        "cost": "Free", "source": "general"},
            {"name": "Public museum",      "description": "Free or discounted entry on select days.",        "cost": "Free-$5", "source": "general"},
        ]


# ── Agent C — Budgeter (GPT-4o, budget-constrained) ─────────
def agent_c(origin, destination, cost, acts, max_budget=None, trip_days=4):
    """
    Synthesizes Agent A + B data into a daily budget breakdown.
    Budget-aware: compares against max_budget and gives actionable advice.
    """
    acts_text = "\n".join(
        f"- {a['name']}: {a['description']} ({a.get('cost','?')})" for a in acts
    )

    budget_constraint = ""
    if max_budget and max_budget > 0:
        budget_constraint = (
            f"\nThe traveler's MAX daily budget is ${max_budget:.0f}. "
            f"Trip duration: {trip_days} days. "
            "Total trip budget = max_daily x trip_days. "
            "If grand_daily_total EXCEEDS max budget, set over_budget=true and provide "
            "specific cost-cutting tips in savings_tips (array of strings). "
            "If within budget, set over_budget=false and savings_tips=[]."
        )

    resp = openai_client.chat.completions.create(
        model="gpt-4o",
        temperature=0.2,
        max_tokens=500,
        messages=[
            {"role": "system", "content": (
                "You are a travel budget analyst. "
                "Given Agent A cost data and Agent B activities, compute a realistic solo daily budget. "
                f"{budget_constraint}\n"
                "Reply ONLY with valid JSON — no markdown:\n"
                '{"daily_hotel":int,"daily_food":int,"daily_transport":int,"daily_activities":int,'
                '"daily_total":int,"flight_amortized":int,"grand_daily_total":int,'
                '"score":int,"summary":"string","over_budget":bool,'
                '"savings_tips":["string"],"total_trip_cost":int}\n'
                f"score: 1-10 (10 = cheapest destination). "
                f"grand_daily_total = daily_total + flight_amortized. "
                f"flight_amortized = round-trip flight midpoint divided by {trip_days}. "
                f"total_trip_cost = daily_total x {trip_days} + flight midpoint."
            )},
            {"role": "user", "content": (
                f"Trip: {origin} → {destination} ({trip_days} days)\n\n"
                f"AGENT A — Cost Data:\n"
                f"  Flight: ${cost['flight_low']}–${cost['flight_high']} round-trip\n"
                f"  Hotel:  ${cost['hotel_low']}–${cost['hotel_high']} / night\n"
                f"  Tip: {cost['booking_tip']}\n\n"
                f"AGENT B — Activities:\n{acts_text}"
            )},
        ],
    )

    raw = resp.choices[0].message.content.strip()
    try:
        result = parse_json(raw)
        result.setdefault("over_budget", False)
        result.setdefault("savings_tips", [])
        result.setdefault("total_trip_cost", 0)
        return result
    except Exception:
        mid_hotel  = (cost["hotel_low"]  + cost["hotel_high"])  // 2
        mid_flight = (cost["flight_low"] + cost["flight_high"]) // 2
        daily      = mid_hotel + 65
        amort      = mid_flight // trip_days
        grand      = daily + amort
        over       = bool(max_budget and grand > max_budget)
        return {
            "daily_hotel": mid_hotel, "daily_food": 40, "daily_transport": 15,
            "daily_activities": 10,   "daily_total": daily,
            "flight_amortized": amort, "grand_daily_total": grand,
            "score": 6, "summary": f"{destination} offers moderate value for budget travelers.",
            "over_budget": over, "savings_tips": [], "total_trip_cost": daily * trip_days + mid_flight,
        }

print("✅ All 3 agents defined (RAG-enhanced + budget-aware).")

✅ All 3 agents defined (RAG-enhanced + budget-aware).


In [ ]:
# CELL 5 — HTML Report Builder (Enhanced with Budget Gauge + RAG Badge)

def build_report(origin, destination, interest, cost, acts, budget,
                 max_budget=None, trip_days=4, rag_chunks=0):
    score       = budget["score"]
    score_color = "#1D9E75" if score >= 7 else "#BA7517" if score >= 5 else "#C0392B"
    score_pct   = score * 10
    over_budget = budget.get("over_budget", False)

    # ── Budget warning banner ──
    budget_banner = ""
    if max_budget and max_budget > 0:
        grand = budget["grand_daily_total"]
        total_trip = budget.get("total_trip_cost", grand * trip_days)
        if over_budget:
            diff = grand - max_budget
            budget_banner = f"""
            <div style='background:#FFF3CD;border:1px solid #FFC107;border-radius:8px;
                        padding:12px 16px;margin-bottom:14px'>
              <div style='font-weight:700;color:#856404;font-size:14px'>
                ⚠️ Over Budget by ${diff:.0f}/day
              </div>
              <div style='font-size:12px;color:#856404;margin-top:4px'>
                Daily total: ${grand} vs your max: ${max_budget:.0f}
                &nbsp;|&nbsp; Total trip ({trip_days}d): ~${total_trip:,}
              </div>
            </div>"""
        else:
            remaining = max_budget - grand
            budget_banner = f"""
            <div style='background:#D4EDDA;border:1px solid #28A745;border-radius:8px;
                        padding:12px 16px;margin-bottom:14px'>
              <div style='font-weight:700;color:#155724;font-size:14px'>
                ✅ Within Budget — ${remaining:.0f}/day to spare
              </div>
              <div style='font-size:12px;color:#155724;margin-top:4px'>
                Daily total: ${grand} vs your max: ${max_budget:.0f}
                &nbsp;|&nbsp; Total trip ({trip_days}d): ~${total_trip:,}
              </div>
            </div>"""

    # ── Savings tips ──
    tips_html = ""
    tips = budget.get("savings_tips", [])
    if tips:
        tip_items = "".join(f"<li style='margin-bottom:4px'>{t}</li>" for t in tips)
        tips_html = f"""
        <div style='background:#FFF8E1;border-radius:8px;padding:10px 14px;margin-top:10px;
                    border:1px solid #FFE082;font-size:13px;color:#5D4037'>
          <div style='font-weight:700;margin-bottom:6px'>💰 Money-Saving Tips:</div>
          <ul style='margin:0;padding-left:20px'>{tip_items}</ul>
        </div>"""

    # ── Agent A budget warning ──
    agent_a_warning = ""
    bw = cost.get("budget_warning")
    if bw and str(bw).lower() not in ("null", "none", ""):
        agent_a_warning = f"<div style='font-size:12px;color:#C0392B;margin-top:6px;font-weight:600'>⚠️ {bw}</div>"

    # ── RAG badge ──
    rag_badge = ""
    if rag_chunks > 0:
        rag_badge = f"""
        <div style='display:inline-block;background:#E8F5E9;border:1px solid #66BB6A;
                    border-radius:20px;padding:3px 12px;font-size:11px;color:#2E7D32;
                    font-weight:600;margin-left:8px'>
          🌐 RAG: {rag_chunks} web chunks indexed
        </div>"""

    act_rows = "".join(f"""
        <tr>
          <td style='padding:7px 0;font-weight:600;color:#1a1a1a;width:30%'>{a['name']}</td>
          <td style='padding:7px 0;color:#444;font-size:13px'>{a['description']}</td>
          <td style='padding:7px 0;text-align:right;color:#185FA5;font-weight:600;white-space:nowrap'>{a.get('cost','?')}</td>
        </tr>
        <tr><td colspan='3' style='border-bottom:1px solid #eee;padding:0'></td></tr>
    """ for a in acts)

    metric_cards = "".join(f"""
        <div style='background:#fff;border:1px solid #e0e0e0;border-radius:8px;
                    padding:10px 14px;text-align:center;flex:1;min-width:80px'>
          <div style='font-size:11px;color:#888;margin-bottom:2px'>{label}</div>
          <div style='font-size:17px;font-weight:700;color:#1a1a1a'>${val}</div>
        </div>
    """ for label, val in [
        ("Hotel",       budget["daily_hotel"]),
        ("Food",        budget["daily_food"]),
        ("Transport",   budget["daily_transport"]),
        ("Activities",  budget["daily_activities"]),
        (f"Flight ÷{trip_days}d",  budget["flight_amortized"]),
    ])

    max_budget_display = f" &nbsp;·&nbsp; Max ${max_budget:.0f}/day" if max_budget else ""

    return f"""
<div style='font-family:system-ui,-apple-system,sans-serif;max-width:700px;
            margin:0 auto;border:1px solid #ddd;border-radius:14px;
            overflow:hidden;box-shadow:0 4px 20px rgba(0,0,0,0.08)'>

  <div style='background:linear-gradient(135deg,#185FA5,#0d3f6e);color:#fff;padding:1.5rem 1.75rem'>
    <div style='display:flex;align-items:center;gap:8px'>
      <span style='font-size:11px;opacity:0.75;letter-spacing:2px;text-transform:uppercase'>Budget Travel Scout</span>
      {rag_badge}
    </div>
    <div style='font-size:26px;font-weight:700;margin-top:4px'>{origin} → {destination}</div>
    <div style='font-size:14px;opacity:0.85;margin-top:4px'>
      Interest: {interest} &nbsp;·&nbsp; {trip_days} days{max_budget_display}
    </div>
  </div>

  <div style='padding:1.25rem 1.75rem;border-bottom:1px solid #eee;background:#f0f7ff'>
    <div style='display:flex;align-items:center;gap:8px;margin-bottom:10px'>
      <span style='background:#185FA5;color:#fff;font-size:11px;font-weight:700;
                   padding:3px 9px;border-radius:20px'>AGENT A</span>
      <span style='font-size:12px;color:#555'>Researcher · GPT-4o + RAG</span>
    </div>
    <div style='display:flex;gap:2.5rem'>
      <div>
        <div style='font-size:11px;color:#888;margin-bottom:2px'>Round-trip flight</div>
        <div style='font-size:22px;font-weight:700;color:#1a1a1a'>${cost['flight_low']}–${cost['flight_high']}</div>
      </div>
      <div>
        <div style='font-size:11px;color:#888;margin-bottom:2px'>Hotel / night</div>
        <div style='font-size:22px;font-weight:700;color:#1a1a1a'>${cost['hotel_low']}–${cost['hotel_high']}</div>
      </div>
    </div>
    <div style='font-size:12px;color:#555;margin-top:8px;font-style:italic'>💡 {cost['booking_tip']}</div>
    {agent_a_warning}
  </div>

  <div style='padding:1.25rem 1.75rem;border-bottom:1px solid #eee;background:#f0fbf6'>
    <div style='display:flex;align-items:center;gap:8px;margin-bottom:10px'>
      <span style='background:#0F6E56;color:#fff;font-size:11px;font-weight:700;
                   padding:3px 9px;border-radius:20px'>AGENT B</span>
      <span style='font-size:12px;color:#555'>Local Guide · GPT-4o + RAG</span>
    </div>
    <table style='width:100%;border-collapse:collapse'>{act_rows}</table>
  </div>

  <div style='padding:1.25rem 1.75rem;background:#f9f7ff'>
    <div style='display:flex;align-items:center;gap:8px;margin-bottom:12px'>
      <span style='background:#533AB7;color:#fff;font-size:11px;font-weight:700;
                   padding:3px 9px;border-radius:20px'>AGENT C</span>
      <span style='font-size:12px;color:#555'>Budgeter · GPT-4o</span>
    </div>
    {budget_banner}
    <div style='display:flex;gap:8px;flex-wrap:wrap;margin-bottom:14px'>{metric_cards}</div>
    <div style='font-size:22px;font-weight:700;color:#1a1a1a;margin-bottom:12px'>
      Grand Daily Total: <span style='color:{score_color}'>${budget['grand_daily_total']}</span>
    </div>
    <div style='margin-bottom:12px'>
      <div style='display:flex;justify-content:space-between;font-size:12px;color:#555;margin-bottom:5px'>
        <span>Budget Friendliness Score</span>
        <span style='font-weight:700;color:{score_color}'>{score} / 10</span>
      </div>
      <div style='height:10px;background:#e0e0e0;border-radius:5px;overflow:hidden'>
        <div style='height:100%;width:{score_pct}%;background:{score_color};border-radius:5px'></div>
      </div>
    </div>
    <div style='font-size:13px;color:#444;line-height:1.7;background:#fff;
                border-radius:8px;padding:10px 14px;border:1px solid #e0e0e0'>
      {budget['summary']}
    </div>
    {tips_html}
  </div>
</div>"""

print("✅ Report builder ready (with budget gauge + RAG badges).")

✅ Report builder ready (with budget gauge + RAG badges).


In [ ]:
# CELL 6 — Launch Gradio UI (Enhanced with Budget + Trip Duration + RAG)
import gradio as gr
import traceback

INTERESTS = ["History", "Nightlife", "Food & Drink", "Nature", "Art & Culture",
             "Adventure", "Shopping", "Beach", "Architecture", "Wildlife"]

def run_pipeline(origin, destination, interest, max_budget, trip_days, progress=gr.Progress()):
    if not origin.strip() or not destination.strip():
        return "", "", "", "<p style='color:red;font-weight:600'>⚠️ Please enter both origin and destination.</p>"

    max_budget = float(max_budget) if max_budget else None
    trip_days  = int(trip_days) if trip_days else 4

    # ── Step 0: RAG — Scrape & index (non-fatal if it fails) ──
    rag_chunks = 0
    try:
        progress(0.05, desc="🌐 Scraping web data for RAG...")
        rag_chunks = build_rag_knowledge(destination.strip())
    except Exception as e:
        print(f"⚠️ RAG build failed (continuing without RAG): {e}")
        traceback.print_exc()

    # ── Step 1: Agent A — Researcher ──
    a_out = ""
    cost = None
    try:
        progress(0.25, desc="Agent A: researching prices...")
        cost = agent_a(origin.strip(), destination.strip(), max_budget)
        a_out = (
            f"✅ Flight (round-trip): ${cost['flight_low']}–${cost['flight_high']}\n"
            f"✅ Hotel (per night):   ${cost['hotel_low']}–${cost['hotel_high']}\n"
            f"💡 {cost['booking_tip']}"
        )
        bw = cost.get("budget_warning")
        if bw and str(bw).lower() not in ("null", "none", ""):
            a_out += f"\n⚠️ {bw}"
    except Exception as e:
        a_out = f"⚠️ Agent A error: {e}"
        print(f"Agent A failed: {e}")
        traceback.print_exc()
        # Provide fallback cost data so B and C can still run
        cost = {
            "flight_low": 300, "flight_high": 800,
            "hotel_low": 70, "hotel_high": 160,
            "booking_tip": "Book 6-8 weeks in advance for best rates.",
            "budget_warning": None,
        }

    # ── Step 2: Agent B — Local Guide ──
    b_out = ""
    acts = None
    try:
        progress(0.50, desc="Agent B: finding activities...")
        acts = agent_b(destination.strip(), interest, max_budget)
        if not acts or len(acts) == 0:
            raise ValueError("Agent B returned empty activities")
        b_out = "\n".join(
            f"{'✅' if 'free' in a.get('cost','').lower() else '💲'} {a['name']} ({a.get('cost','?')}) — {a['description']}"
            for a in acts
        )
        if not b_out.strip():
            b_out = "⚠️ Agent B returned no formatted output — using fallback"
    except Exception as e:
        b_out = f"⚠️ Agent B error: {e}"
        print(f"Agent B failed: {e}")
        traceback.print_exc()
        # Provide fallback activities so C can still run
        acts = [
            {"name": "City walking tour",  "description": f"Explore {destination}'s landmarks.", "cost": "Free", "source": "fallback"},
            {"name": "Local market visit", "description": "Browse produce, crafts, street food.", "cost": "Free", "source": "fallback"},
            {"name": "Public museum",      "description": "Free entry on select days.",           "cost": "Free-$5", "source": "fallback"},
        ]

    # ── Step 3: Agent C — Budgeter ──
    c_out = ""
    budget = None
    try:
        progress(0.75, desc="Agent C: building budget breakdown...")
        budget = agent_c(origin.strip(), destination.strip(), cost, acts, max_budget, trip_days)
        c_out = (
            f"Hotel ${budget['daily_hotel']} | Food ${budget['daily_food']} | "
            f"Transport ${budget['daily_transport']} | Activities ${budget['daily_activities']}\n"
            f"Flight amortized ({trip_days}d): ${budget['flight_amortized']}\n"
            f"Grand daily total: ${budget['grand_daily_total']}  |  Score: {budget['score']}/10"
        )
        if budget.get("over_budget"):
            c_out += f"\n⚠️ OVER your ${max_budget:.0f}/day budget!"
        if budget.get("total_trip_cost"):
            c_out += f"\n📊 Est. total trip cost ({trip_days}d): ${budget['total_trip_cost']:,}"
    except Exception as e:
        c_out = f"⚠️ Agent C error: {e}"
        print(f"Agent C failed: {e}")
        traceback.print_exc()
        # Minimal fallback budget
        mid_hotel  = (cost["hotel_low"]  + cost["hotel_high"])  // 2
        mid_flight = (cost["flight_low"] + cost["flight_high"]) // 2
        budget = {
            "daily_hotel": mid_hotel, "daily_food": 40, "daily_transport": 15,
            "daily_activities": 10, "daily_total": mid_hotel + 65,
            "flight_amortized": mid_flight // trip_days,
            "grand_daily_total": mid_hotel + 65 + mid_flight // trip_days,
            "score": 5, "summary": f"Budget estimate for {destination} (fallback).",
            "over_budget": False, "savings_tips": [], "total_trip_cost": 0,
        }

    progress(1.0, desc="Done!")
    try:
        report = build_report(origin, destination, interest, cost, acts, budget,
                              max_budget, trip_days, rag_chunks)
    except Exception as e:
        report = f"<p style='color:red'><strong>Report build error:</strong> {e}</p>"

    return a_out, b_out, c_out, report


with gr.Blocks(
    title="Budget Travel Scout",
    theme=gr.themes.Soft(primary_hue="blue", neutral_hue="slate"),
    css="""
        .agent-box textarea { font-size:13px !important; font-family:monospace !important; }
        footer { display:none !important; }
    """,
) as demo:

    gr.HTML("""
        <div style='text-align:center;padding:1.5rem 0 0.5rem'>
          <h1 style='font-size:28px;font-weight:700;color:#185FA5;margin:0'>🌍 Budget Travel Scout</h1>
          <p style='color:#666;margin-top:6px;font-size:14px'>
            Multi-Agent AI with <b>RAG</b> (Web Scraping → ChromaDB → LLM)
          </p>
          <p style='color:#999;margin-top:2px;font-size:12px'>
            <b>Agent A</b>: GPT-4o + RAG (Researcher) &nbsp;·&nbsp;
            <b>Agent B</b>: GPT-4o + RAG (Local Guide) &nbsp;·&nbsp;
            <b>Agent C</b>: GPT-4o (Budgeter)
          </p>
        </div>
    """)

    with gr.Row():
        origin_in      = gr.Textbox(label="✈️ From (city)", placeholder="e.g. Boston", scale=2)
        destination_in = gr.Textbox(label="📍 To (destination)", placeholder="e.g. Lisbon", scale=2)
        interest_in    = gr.Dropdown(choices=INTERESTS, value="History", label="🎯 Interest", scale=1)

    with gr.Row():
        max_budget_in  = gr.Number(label="💰 Max Daily Budget (USD)", value=None,
                                    minimum=0, maximum=5000,
                                    info="Optional — leave empty for no limit")
        trip_days_in   = gr.Slider(label="📅 Trip Duration (days)", minimum=1, maximum=30,
                                    value=4, step=1)

    run_btn = gr.Button("🔍 Scout this destination", variant="primary", size="lg")

    gr.Markdown("### Agent Outputs")
    with gr.Row():
        with gr.Column(elem_classes="agent-box"):
            gr.Markdown("**🔵 Agent A — Researcher** *(GPT-4o + RAG)*")
            a_out = gr.Textbox(label="", lines=5, interactive=False, placeholder="Waiting...")
        with gr.Column(elem_classes="agent-box"):
            gr.Markdown("**🟢 Agent B — Local Guide** *(GPT-4o + RAG)*")
            b_out = gr.Textbox(label="", lines=5, interactive=False, placeholder="Waiting...")
        with gr.Column(elem_classes="agent-box"):
            gr.Markdown("**🟣 Agent C — Budgeter** *(GPT-4o)*")
            c_out = gr.Textbox(label="", lines=5, interactive=False, placeholder="Waiting...")

    gr.Markdown("### Full Report")
    report_out = gr.HTML("<p style='color:#aaa;text-align:center;padding:2rem'>Run a search to see your travel report.</p>")

    gr.Examples(
        examples=[
            ["Boston",       "Lisbon",       "History",       150, 5],
            ["New York",     "Bangkok",      "Food & Drink",  80,  7],
            ["Chicago",      "Tokyo",        "Nightlife",     200, 4],
            ["Los Angeles",  "Mexico City",  "Art & Culture",  100, 5],
        ],
        inputs=[origin_in, destination_in, interest_in, max_budget_in, trip_days_in],
        label="Try an example",
    )

    gr.HTML("""
        <div style='text-align:center;padding:1rem 0;color:#aaa;font-size:12px'>
          RAG Pipeline: BeautifulSoup (scrape) → SentenceTransformers (embed) → ChromaDB (retrieve) → GPT-4o (generate)
        </div>
    """)

    run_btn.click(
        fn=run_pipeline,
        inputs=[origin_in, destination_in, interest_in, max_budget_in, trip_days_in],
        outputs=[a_out, b_out, c_out, report_out],
    )

demo.launch(share=True)

/tmp/ipykernel_8583/31589157.py:117: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_8583/31589157.py:117: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5ae6f4f74bcfa91c48.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
